# 09 - Self-RAG（自反思 RAG）

**复杂度：** ⭐⭐⭐⭐⭐

**应用场景：** 探索性研究、对质量要求高的应用、自校正系统

**核心特性：**
- LLM 自主决定何时进行检索
- 自我批评机制
- 迭代优化
- 引用验证

**流程：**
```
查询 → 需要检索吗？→ 
  是：检索 + 生成 → 自我批评 → 如果质量差则重试
  否：直接生成 → 自我批评
```

In [1]:
import sys

sys.path.append("../..")

from shared.config import create_chat_model, create_embeddings, DEFAULT_MODEL, DEFAULT_TEMPERATURE, DEFAULT_VISION_MODEL
from shared.config import HF_VECTOR_STORE_PATH, DEFAULT_MODEL
from shared.utils import load_vector_store, print_section_header, format_docs
from shared.prompts import (
    RETRIEVAL_NEED_PROMPT,
    SELF_CRITIQUE_PROMPT,
    CITATION_CHECK_PROMPT,
)
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

print_section_header("Setup: Self-RAG")

embeddings = create_embeddings()
vectorstore = load_vector_store(HF_VECTOR_STORE_PATH, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
llm = create_chat_model(model=DEFAULT_MODEL, temperature=0)

print("✅ Setup complete!")


设置：Self-RAG



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded vector store from d:\ZKS_Data\AI\langchain-rag-tutorial\notebooks\advanced_architectures\..\..\data\vector_stores\huggingface_embeddings


langchain-openai injected a custom httpx transport to apply `http_socket_options`, which disables httpx's proxy auto-detection (system proxy configuration detected). Set `LANGCHAIN_OPENAI_TCP_KEEPALIVE=0` or pass `http_socket_options=()` to restore default proxy behavior, or supply `openai_proxy` / your own `http_client` / `http_async_client` to take full control.


✅ 设置完成！


## 2. Self-RAG 组件

In [2]:
print_section_header("Self-RAG Components")

# 检索需求决策器
retrieval_decider = RETRIEVAL_NEED_PROMPT | llm | StrOutputParser()

# 响应批评器
response_critic = SELF_CRITIQUE_PROMPT | llm | StrOutputParser()

# 引用检查器
citation_checker = CITATION_CHECK_PROMPT | llm | StrOutputParser()

print("✓ Self-RAG components initialized:")
print("  - Retrieval need decider")
print("  - Response critic")
print("  - Citation checker")


SELF-RAG 组件

✓ Self-RAG 组件已初始化：
  - 检索需求决策器
  - 响应批评器
  - 引用检查器


## 3. 测试组件

In [3]:
print_section_header("Testing Self-RAG Components")

# 测试检索决策
queries = [
    "What is 2+2?",  # NO retrieval
    "What is RAG in LangChain?",  # YES retrieval
]

print("Retrieval Need Decisions:\n")
for q in queries:
    decision = retrieval_decider.invoke({"query": q}).strip()
    print(f"{decision:3} | {q}")


测试 Self-RAG 组件

检索需求决策：

NO  | What is 2+2?
NO  | What is RAG in LangChain?


## 4. Self-RAG 流水线

In [4]:
print_section_header("Self-RAG Pipeline")

def self_rag_pipeline(query: str, max_iterations: int = 2):
    """Self-RAG with iterative refinement."""
    print(f"\n{'='*80}")
    print(f"Query: {query}")
    print('='*80)
    
    iteration = 0
    
    while iteration < max_iterations:
        iteration += 1
        print(f"\n--- Iteration {iteration} ---")
        
        # Decide if retrieval needed
        need_retrieval = retrieval_decider.invoke({"query": query})
        print(f"Retrieval needed: {need_retrieval.strip()}")
        
        # Retrieve or use general knowledge
        if "YES" in need_retrieval.upper():
            docs = retriever.invoke(query)
            context = format_docs(docs)
            print(f"Retrieved {len(docs)} documents")
        else:
            context = "Using general knowledge."
            print("Using general knowledge only")
        
        # Generate response
        gen_prompt = ChatPromptTemplate.from_messages([
            ("system", f"Context: {context}"),
            ("user", "{query}")
        ])
        response = (gen_prompt | llm | StrOutputParser()).invoke({"query": query})
        print(f"\nGenerated ({len(response)} chars)")
        
        # Self-critique
        critique = response_critic.invoke({
            "query": query,
            "context": context[:1000],
            "response": response
        })
        
        print(f"\n🔍 Critique:\n{critique}")
        
        # Check if retry needed
        if "SHOULD_RETRY: yes" not in critique.lower():
            print("\n✓ Response approved!")
            return response, iteration, critique
        else:
            print("\n⚠️  Retrying with refinement...")
    
    print("\n⚠️  Max iterations reached")
    return response, iteration, critique

print("✓ Self-RAG pipeline ready")


SELF-RAG 流水线

✓ Self-RAG 流水线已就绪


## 5. 测试 Self-RAG

In [5]:
print_section_header("Self-RAG Test")

# 测试 1：需要检索的查询
query1 = "What is the difference between similarity and MMR retrieval?"
response1, iters1, _ = self_rag_pipeline(query1)

print(f"\n\n最终响应 ({iters1} 次迭代{'s' if iters1 > 1 else ''}):")
print(response1[:300])


SELF-RAG 测试


Query: What is the difference between similarity and MMR retrieval?

--- Iteration 1 ---
Retrieval needed: NO
Using general knowledge only

Generated (5932 chars)

🔍 Critique:
SCORE: 5
ISSUES: None
SHOULD_RETRY: no

✓ Response approved!


Final Response (1 iteration):
This is an excellent question that hits at a core distinction in information retrieval, especially within modern AI applications like RAG (Retrieval-Augmented Generation).

Here's the breakdown of the difference between **Similarity Search** and **MMR (Maximal Marginal Relevance) Retrieval**.

### T


In [6]:
# 测试 2：简单查询（无需检索）
query2 = "What is 5 + 7?"
response2, iters2, _ = self_rag_pipeline(query2)

print(f"\n\n最终响应 ({iters2} 次迭代{'s' if iters2 > 1 else ''}):")
print(response2)


Query: What is 5 + 7?

--- Iteration 1 ---
Retrieval needed: NO
Using general knowledge only

Generated (16 chars)

🔍 Critique:
SCORE: 5
ISSUES: None
SHOULD_RETRY: no

✓ Response approved!


Final Response (1 iteration):
5 + 7 equals 12.


## 总结

**优势：**
✅ 自主决策能力  
✅ 自我校正能力  
✅ 仅在需要时检索（高效）  
✅ 内置质量保证  

**局限性：**
- 非常慢（多次 LLM 调用）
- 成本高（迭代 + 批评）
- 调优复杂
- 可能过度校正

**适用场景：**
- 质量至关重要
- 研究应用
- 自校正系统
- 预算允许较高成本

**生产环境建议：**
- 限制最大迭代次数
- 缓存批评结果
- 监控迭代分布
- 设置质量阈值

**下一步：** [10_agentic_rag.ipynb](10_agentic_rag.ipynb) - 带工具的自主智能体